In [1]:
import os
import zipfile
from google.colab import drive

drive.mount('/content/drive')

os.makedirs('/content/datasetbanknot', exist_ok=True)
with zipfile.ZipFile('/content/drive/MyDrive/data_banknot.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/datasetbanknot')

os.makedirs('/content/datasetgizem', exist_ok=True)
with zipfile.ZipFile('/content/drive/MyDrive/datasetgizem.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/datasetgizem')

print("Zip dosyalari cikarildi.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Zip dosyalari cikarildi.


In [2]:
import shutil

mapping = {'0': '1', '1': '0', '2': '3', '3': '2', '4': '5', '5': '4'}

def fix_gizem_labels(base_path, mapping):
    for split in ['train', 'valid', 'test', 'val']:
        label_dir = os.path.join(base_path, split, 'labels')
        if not os.path.exists(label_dir):
            continue
        for filename in os.listdir(label_dir):
            if filename.endswith('.txt'):
                filepath = os.path.join(label_dir, filename)
                with open(filepath, 'r') as f:
                    lines = f.readlines()
                new_lines = []
                for line in lines:
                    parts = line.split()
                    if parts and parts[0] in mapping:
                        parts[0] = mapping[parts[0]]
                        new_lines.append(" ".join(parts) + "\n")
                with open(filepath, 'w') as f:
                    f.writelines(new_lines)

fix_gizem_labels('/content/datasetgizem', mapping)

def merge_datasets(source_folders, target_folder):
    target_splits = ['train', 'valid', 'test']
    subdirs = ['images', 'labels']
    for split in target_splits:
        for subdir in subdirs:
            os.makedirs(os.path.join(target_folder, split, subdir), exist_ok=True)
    for src in source_folders:
        if not os.path.exists(src):
            continue
        current_source_splits = os.listdir(src)
        for s_split in current_source_splits:
            t_split = 'valid' if s_split == 'val' else s_split
            if t_split not in target_splits:
                continue
            src_img_path = os.path.join(src, s_split, 'images')
            if os.path.exists(src_img_path):
                for file_name in os.listdir(src_img_path):
                    shutil.copy(os.path.join(src_img_path, file_name), os.path.join(target_folder, t_split, 'images', file_name))
                    label_name = os.path.splitext(file_name)[0] + '.txt'
                    src_label_full = os.path.join(src, s_split, 'labels', label_name)
                    if os.path.exists(src_label_full):
                        shutil.copy(src_label_full, os.path.join(target_folder, t_split, 'labels', label_name))

if os.path.exists('/content/final_dataset'):
    shutil.rmtree('/content/final_dataset')

sources = ['/content/datasetbanknot', '/content/datasetgizem']
destination = '/content/final_dataset'
merge_datasets(sources, destination)

print("Veriler birlestirildi ve etiketler duzeltildi.")

Veriler birlestirildi ve etiketler duzeltildi.


In [3]:
import yaml

data_config = {
    'path': '/content/final_dataset',
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'nc': 6,
    'names': ['100', '10', '200', '20', '50', '5']
}

with open('/content/final_dataset/data.yaml', 'w') as f:
    yaml.dump(data_config, f)

print("data.yaml kaydedildi.")

data.yaml kaydedildi.


In [4]:
from ultralytics import YOLO

model = YOLO('yolov8s.pt')

results = model.train(
    data='/content/final_dataset/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    cls=2.5,
    hsv_s=0.0,
    hsv_h=0.02,
    patience=25,
    dropout=0.2,
    close_mosaic=20,
    project='banknot_projesi_final',
    name='biktim_model',
    save=True
)

Ultralytics 8.3.241 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: task=detect, mode=train, model=yolov8s.pt, data=/content/final_dataset/data.yaml, epochs=100, patience=25, batch=16, imgsz=640, save=True, project=banknot_projesi_final, name=biktim_model, exist_ok=False, dropout=0.2, val=True
Overriding model.yaml nc=80 with nc=6

                   from  n    params  module                                       arguments                     
  0                  -1  1       928  ultralytics.nn.modules.conv.Conv             [3, 32, 3, 2]                 
  1                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  2                  -1  1     29056  ultralytics.nn.modules.block.C2f             [64, 64, 1, True]             
  3                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  4                  -1  2    197632  ultralytics.nn.modules.block.

In [5]:
from ultralytics import YOLO

model = YOLO('/content/banknot_projesi_final/biktim_model/weights/best.pt')
model.export(format='tflite', imgsz=640)

Ultralytics 8.3.241 Python-3.12.12 torch-2.9.0+cu126 CPU (Intel Xeon CPU @ 2.20GHz)
Model summary (fused): 72 layers, 11,127,906 parameters, 0 gradients, 28.4 GFLOPs
PyTorch: starting from '/content/banknot_projesi_final/biktim_model/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 10, 8400) (21.5 MB)
TensorFlow SavedModel: starting export with tensorflow 2.19.0...
ONNX: starting export with onnx 1.20.0 opset 22...
ONNX: slimming with onnxslim 0.1.80...
ONNX: export success 2.4s, saved as '/content/banknot_projesi_final/biktim_model/weights/best.onnx' (42.8 MB)
TensorFlow SavedModel: starting TFLite export with onnx2tf 1.28.8...
TensorFlow SavedModel: export success 38.3s, saved as '/content/banknot_projesi_final/biktim_model/weights/best_saved_model' (107.0 MB)
TensorFlow Lite: starting export with tensorflow 2.19.0...
TensorFlow Lite: export success 0.0s, saved as '/content/banknot_projesi_final/biktim_model/weights/best_saved_model/best_float32.tflite'

In [6]:
import shutil
import os

hedef_klasor = '/content/drive/MyDrive/YOLO_Modellerim'
kaynak_dosya = '/content/banknot_projesi_final/biktim_model/weights/best.pt'
yeni_dosya_adi = f'{hedef_klasor}/biktim_model_best.pt'

if not os.path.exists(hedef_klasor):
    os.makedirs(hedef_klasor)

try:
    shutil.copy(kaynak_dosya, yeni_dosya_adi)
    print(f"Model Kaydedildi: {yeni_dosya_adi}")
except FileNotFoundError:
    print("Hata: Kaynak dosya bulunamadi!")

kaynak_klasor_tam = '/content/banknot_projesi_final/biktim_model'
hedef_klasor_tam = '/content/drive/MyDrive/YOLO_Egitim_Sonuclari/biktim_model'

if os.path.exists(kaynak_klasor_tam):
    shutil.copytree(kaynak_klasor_tam, hedef_klasor_tam, dirs_exist_ok=True)
    print(f"Tum sonuclar kopyalandi: {hedef_klasor_tam}")

kaynak_tflite_klasoru = '/content/banknot_projesi_final/biktim_model/weights/best_saved_model'
hedef_drive_tflite_yolu = '/content/drive/MyDrive/YOLO_TFLite_Model_Yedek'

if os.path.exists(kaynak_tflite_klasoru):
    shutil.copytree(kaynak_tflite_klasoru, hedef_drive_tflite_yolu, dirs_exist_ok=True)
    print(f"TFLite model klasoru kaydedildi: {hedef_drive_tflite_yolu}")

Model Kaydedildi: /content/drive/MyDrive/YOLO_Modellerim/biktim_model_best.pt
Tum sonuclar kopyalandi: /content/drive/MyDrive/YOLO_Egitim_Sonuclari/biktim_model
TFLite model klasoru kaydedildi: /content/drive/MyDrive/YOLO_TFLite_Model_Yedek


In [7]:
from ultralytics import YOLO

model = YOLO('/content/banknot_projesi_final/biktim_model/weights/best.pt')
results = model.predict('/content/drive/MyDrive/a.jpg', conf=0.1)

toplam_tutar = 0

for r in results:
    if len(r.boxes) == 0:
        print("Gorselde tanimli herhangi bir para bulunamadi.")
    else:
        print("Tespit Sonuclari:")
        for box in r.boxes:
            class_id = int(box.cls[0])
            label = model.names[class_id]
            conf = float(box.conf[0])
            
            try:
                sadece_sayi = "".join(filter(str.isdigit, label))
                deger = int(sadece_sayi)
                toplam_tutar += deger
            except ValueError:
                deger = 0
            
            print(f"Tespit: {label} TL (Dogruluk Payi: %{conf*100:.1f})")

print("-" * 30)
if toplam_tutar > 0:
    print(f"GORSELDEKI TOPLAM TUTAR: {toplam_tutar} TL")
else:
    print("Hesaplanacak bir tutar bulunamadi.")
print("-" * 30)

image 1/1 /content/drive/MyDrive/a.jpg: 640x640 2 100s, 1 200, 1 20, 1 50, 1 5, 16.2ms
Speed: 1.5ms preprocess, 16.2ms inference, 1.3ms postprocess per image at shape (1, 3, 640, 640)
Tespit Sonuclari:
Tespit: 20 TL (Dogruluk Payi: %94.1)
Tespit: 100 TL (Dogruluk Payi: %92.7)
Tespit: 5 TL (Dogruluk Payi: %92.3)
Tespit: 50 TL (Dogruluk Payi: %92.2)
Tespit: 200 TL (Dogruluk Payi: %79.6)
Tespit: 100 TL (Dogruluk Payi: %65.9)
------------------------------
GORSELDEKI TOPLAM TUTAR: 475 TL
------------------------------
